In [16]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("../data/processed/features.csv")
df = df.sort_values(['Email', 'assignment_order']).reset_index(drop=True)

print(f"Loaded: {df.shape}")
print(f"Students: {df['Email'].nunique()}")
print(f"At-risk balance: {df.groupby('Email')['at_risk'].first().value_counts().to_dict()}")

Loaded: (588, 18)
Students: 42
At-risk balance: {1: 21, 0: 21}


In [17]:
# WHY THIS MATTERS:
# Normal train/test split randomly shuffles rows.
# If we do that here, the model trains on row 5 of student A
# and tests on row 3 of student A — it already "knows" that student!
# This is called DATA LEAKAGE and produces fake high accuracy.
#
# Correct approach: split by STUDENT.
# Training students: never appear in test set.
# Test students: model has never seen any of their rows.

# Get unique student emails
students = df['Email'].unique()
print(f"Total students: {len(students)}")

# 80% train, 20% test = 33 train, 9 test students
np.random.seed(42)  # seed for reproducibility
np.random.shuffle(students)

split_idx = int(len(students) * 0.8)
train_students = students[:split_idx]
test_students  = students[split_idx:]

print(f"Train students: {len(train_students)}")
print(f"Test students:  {len(test_students)}")

# Create train/test DataFrames
df_train = df[df['Email'].isin(train_students)].copy()
df_test  = df[df['Email'].isin(test_students)].copy()

print(f"\nTrain rows: {len(df_train)}")
print(f"Test rows:  {len(df_test)}")
print(f"\nAt-risk in train: {df_train.groupby('Email')['at_risk'].first().mean()*100:.1f}%")
print(f"At-risk in test:  {df_test.groupby('Email')['at_risk'].first().mean()*100:.1f}%")

Total students: 42
Train students: 33
Test students:  9

Train rows: 462
Test rows:  126

At-risk in train: 54.5%
At-risk in test:  33.3%


In [18]:
# These are the 13 numeric features the model sees
FEATURE_COLS = [
    'grade_pct', 'time_minutes', 'never_started',
    'grade_zscore', 'time_zscore', 'effort_efficiency',
    'class_skip_rate', 'grade_velocity', 'time_velocity',
    'rolling_avg_grade_3', 'cumulative_skips',
    'cumulative_avg_grade', 'difficulty_score'
]

# Classification targets (at risk yes/no)
X_train_clf = df_train[FEATURE_COLS].values
y_train_clf = df_train['at_risk'].values

X_test_clf  = df_test[FEATURE_COLS].values
y_test_clf  = df_test['at_risk'].values

# Regression targets (what score on next assignment)
X_train_reg = df_train[FEATURE_COLS].values
y_train_reg = df_train['next_grade'].values

X_test_reg  = df_test[FEATURE_COLS].values
y_test_reg  = df_test['next_grade'].values

# Feature scaling — WHY:
# Logistic Regression is sensitive to feature scale.
# grade_pct ranges 0-100, time_minutes might be 0-500.
# Without scaling, large-range features dominate.
# StandardScaler converts each feature to mean=0, std=1.
# Random Forest does NOT need scaling (tree-based models
# make splits based on rank order, not absolute values)
# but we scale anyway for consistency.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clf)
X_test_scaled  = scaler.transform(X_test_clf)
# Note: fit_transform on TRAIN only, then transform on TEST
# If we fit on test data too, we'd be leaking test statistics

print("Feature matrix shapes:")
print(f"  X_train: {X_train_clf.shape}")
print(f"  X_test:  {X_test_clf.shape}")
print(f"\ny_train at-risk balance: {np.bincount(y_train_clf)}")
print(f"y_test  at-risk balance: {np.bincount(y_test_clf)}")

Feature matrix shapes:
  X_train: (462, 13)
  X_test:  (126, 13)

y_train at-risk balance: [210 252]
y_test  at-risk balance: [84 42]


In [19]:
# Logistic Regression is the simplest possible classifier.
# It learns one weight per feature — a straight line decision boundary.
# If our GRU can't beat this, the temporal modeling adds no value.

# class_weight='balanced' tells sklearn to automatically give more
# importance to the minority class during training
# (compensates for any remaining imbalance)

mlflow.set_experiment("academic_performance_baselines")

with mlflow.start_run(run_name="logistic_regression"):
    
    lr = LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    )
    lr.fit(X_train_scaled, y_train_clf)
    y_pred_lr = lr.predict(X_test_scaled)
    
    # Metrics
    acc  = accuracy_score(y_test_clf, y_pred_lr)
    
    # Log to MLflow
    mlflow.log_param("model", "logistic_regression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("accuracy", acc)
    mlflow.sklearn.log_model(lr, "model")
    
    print("=" * 50)
    print("LOGISTIC REGRESSION — Classification Results")
    print("=" * 50)
    print(f"Accuracy: {acc*100:.1f}%\n")
    print("Detailed breakdown:")
    print(classification_report(y_test_clf, y_pred_lr,
                                  target_names=['Not At Risk', 'At Risk']))
    
    # Feature importance (coefficients)
    print("\nTop features by importance (absolute coefficient):")
    coef_df = pd.DataFrame({
        'feature': FEATURE_COLS,
        'coefficient': np.abs(lr.coef_[0])
    }).sort_values('coefficient', ascending=False)
    print(coef_df.to_string(index=False))

2026/04/09 21:36:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 21:36:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LOGISTIC REGRESSION — Classification Results
Accuracy: 86.5%

Detailed breakdown:
              precision    recall  f1-score   support

 Not At Risk       0.88      0.93      0.90        84
     At Risk       0.84      0.74      0.78        42

    accuracy                           0.87       126
   macro avg       0.86      0.83      0.84       126
weighted avg       0.86      0.87      0.86       126


Top features by importance (absolute coefficient):
             feature  coefficient
cumulative_avg_grade     3.185188
         time_zscore     0.800995
       time_velocity     0.543526
        time_minutes     0.532951
        grade_zscore     0.305020
    difficulty_score     0.181085
      grade_velocity     0.174338
     class_skip_rate     0.173919
           grade_pct     0.154788
    cumulative_skips     0.149711
 rolling_avg_grade_3     0.132897
       never_started     0.124750
   effort_efficiency     0.013796


In [20]:
with mlflow.start_run(run_name="random_forest_classifier"):
    
    rf_clf = RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        max_depth=5
    )
    rf_clf.fit(X_train_clf, y_train_clf)
    y_pred_rf = rf_clf.predict(X_test_clf)
    
    acc_rf = accuracy_score(y_test_clf, y_pred_rf)
    
    mlflow.log_param("model", "random_forest_classifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 5)
    mlflow.log_metric("accuracy", acc_rf)
    mlflow.sklearn.log_model(rf_clf, "model")
    
    print("=" * 50)
    print("RANDOM FOREST — Classification Results")
    print("=" * 50)
    print(f"Accuracy: {acc_rf*100:.1f}%\n")
    print("Detailed breakdown:")
    print(classification_report(y_test_clf, y_pred_rf,
                                  target_names=['Not At Risk', 'At Risk']))
    
    # Feature importance — which features does RF find most useful?
    print("\nFeature importance (Random Forest):")
    fi_df = pd.DataFrame({
        'feature': FEATURE_COLS,
        'importance': rf_clf.feature_importances_
    }).sort_values('importance', ascending=False)
    print(fi_df.to_string(index=False))

2026/04/09 21:37:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 21:37:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RANDOM FOREST — Classification Results
Accuracy: 84.1%

Detailed breakdown:
              precision    recall  f1-score   support

 Not At Risk       0.89      0.87      0.88        84
     At Risk       0.75      0.79      0.77        42

    accuracy                           0.84       126
   macro avg       0.82      0.83      0.82       126
weighted avg       0.84      0.84      0.84       126


Feature importance (Random Forest):
             feature  importance
cumulative_avg_grade    0.233435
         time_zscore    0.145978
 rolling_avg_grade_3    0.136760
        time_minutes    0.126933
        grade_zscore    0.089028
           grade_pct    0.072140
   effort_efficiency    0.062559
       time_velocity    0.047795
      grade_velocity    0.041621
    difficulty_score    0.019098
     class_skip_rate    0.010378
    cumulative_skips    0.010045
       never_started    0.004230


In [21]:
with mlflow.start_run(run_name="random_forest_regressor"):
    
    rf_reg = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        max_depth=5
    )
    rf_reg.fit(X_train_reg, y_train_reg)
    y_pred_reg = rf_reg.predict(X_test_reg)
    
    mae  = mean_absolute_error(y_test_reg, y_pred_reg)
    rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
    r2   = r2_score(y_test_reg, y_pred_reg)
    
    mlflow.log_param("model", "random_forest_regressor")
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.sklearn.log_model(rf_reg, "model")
    
    print("=" * 50)
    print("RANDOM FOREST — Regression Results (next grade)")
    print("=" * 50)
    print(f"MAE  (Mean Absolute Error):  {mae:.2f} points")
    print(f"RMSE (Root Mean Sq Error):   {rmse:.2f} points")
    print(f"R²   (Variance explained):   {r2:.3f}")
    print()
    print("What these numbers mean:")
    print(f"  On average, predictions are off by {mae:.1f} grade points")
    print(f"  A perfect model scores R²=1.0, random guessing scores R²=0.0")

2026/04/09 21:37:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 21:37:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RANDOM FOREST — Regression Results (next grade)
MAE  (Mean Absolute Error):  18.07 points
RMSE (Root Mean Sq Error):   25.36 points
R²   (Variance explained):   0.391

What these numbers mean:
  On average, predictions are off by 18.1 grade points
  A perfect model scores R²=1.0, random guessing scores R²=0.0


In [22]:
print("To view your experiment results in MLflow:")
print()
print("  1. Open a NEW terminal in VS Code")
print("  2. Activate venv: venv\\Scripts\\activate")
print("  3. Run: mlflow ui")
print("  4. Open browser: http://127.0.0.1:5000")
print()
print("You will see all 3 runs with their metrics side by side.")
print()
print("Phase 3 baseline numbers to beat with the GRU:")
print(f"  Classification accuracy:  record what you see above")
print(f"  Regression MAE:           record what you see above")
print()
print("Phase 3 complete.")

To view your experiment results in MLflow:

  1. Open a NEW terminal in VS Code
  2. Activate venv: venv\Scripts\activate
  3. Run: mlflow ui
  4. Open browser: http://127.0.0.1:5000

You will see all 3 runs with their metrics side by side.

Phase 3 baseline numbers to beat with the GRU:
  Classification accuracy:  record what you see above
  Regression MAE:           record what you see above

Phase 3 complete.
